# 4장 실습 ② — 데이터 누수

**TensorFlow 판**

§4.3 ★의 **틀린 코드**와 **맞는 코드**를 나란히 돌립니다.

```python
x = (x - x.mean()) / x.std()   # 틀림 — 전체로 정규화하고
s = data.split(x, y)           #        그다음에 나눈다

s = data.split(x, y)                            # 맞음 — 먼저 나누고
mu, sd = s.x_train.mean(0), s.x_train.std(0)    #        학습 통계로만
```

> **결과를 미리 말씀드립니다. 이 둘은 차이가 안 납니다.**
> 그런데 누수는 무섭습니다. 두 문장이 어떻게 같이 참인지가 이 실습입니다.

## 4.0 준비

In [1]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [2]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

{'python': '3.12.3', 'numpy': '2.1.3', 'keras': '3.15.1', 'tensorflow': '2.21.0', 'torch': '-', 'keras_backend': 'tensorflow'}


## 4.1 학습 함수 — 여기만 판마다 다릅니다

In [3]:
import tensorflow as tf

L_ = tf.keras.layers

def train(s, epochs=40, lr=0.01):
    """같은 모델, 같은 조건. **이 함수만 판마다 다릅니다.**"""
    dlbook.set_seed(42)
    m = tf.keras.Sequential([
        L_.Input(shape=(s.x_train.shape[1],)),
        L_.Dense(16, activation="relu"),
        L_.Dense(1, activation="sigmoid")])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy")
    m.fit(s.x_train, s.y_train, epochs=dlbook.smoke.epochs(epochs),
          batch_size=16, verbose=0)
    pred = (m.predict(s.x_test, verbose=0).ravel() > 0.5).astype(int)
    return metrics.accuracy(s.y_test, pred)

## 4.2 정규화 누수 — 얼마나 부풀려지나

In [4]:
def normalize_wrong(x, y, seed):
    """틀린 순서 — 전체로 정규화하고 나눈다."""
    z = (x - x.mean(0)) / x.std(0)
    return data.split(z, y, seed=seed)


def normalize_right(x, y, seed):
    """맞는 순서 — 나누고, 학습 데이터의 통계로만."""
    s = data.split(x, y, seed=seed)
    mu, sd = s.x_train.mean(0), s.x_train.std(0)
    f = lambda a: (a - mu) / sd
    return data.Split(f(s.x_train), s.y_train,
                      f(s.x_val), s.y_val, f(s.x_test), s.y_test)


print(f"{'데이터 크기':<14}{'누수':>10}{'제대로':>10}{'차이':>10}")
print("-" * 44)
for n in (100, 300, 1000):
    x, y = data.apples(n, seed=1)
    x = x.copy(); x[:, 1] = x[:, 1] * 10000.0
    a_bad = train(normalize_wrong(x, y, 7))
    a_ok = train(normalize_right(x, y, 7))
    print(f"{n:<14,}{a_bad:>10.3f}{a_ok:>10.3f}{a_bad - a_ok:>+10.3f}")
    dlbook.record(f"ch04_leak_norm_gap_n{n}", a_bad - a_ok)

print()
print("★ **차이가 없습니다.** 데이터를 줄여도 마찬가지입니다.")
print("  평균과 표준편차는 표본이 조금만 있어도 안정적이라,")
print("  전체로 재나 학습 데이터로만 재나 거의 같은 값이 나옵니다.")
print()
print("  그러면 §4.3의 '맞는 코드'는 왜 지켜야 합니까. 다음 칸입니다.")

데이터 크기                누수       제대로        차이
--------------------------------------------


100                0.950     0.950    +0.000
ch04_leak_norm_gap_n100 = 0.0000


300                0.900     0.900    +0.000
ch04_leak_norm_gap_n300 = 0.0000


1,000              0.915     0.915    +0.000
ch04_leak_norm_gap_n1000 = 0.0000

★ **차이가 없습니다.** 데이터를 줄여도 마찬가지입니다.
  평균과 표준편차는 표본이 조금만 있어도 안정적이라,
  전체로 재나 학습 데이터로만 재나 거의 같은 값이 나옵니다.

  그러면 §4.3의 '맞는 코드'는 왜 지켜야 합니까. 다음 칸입니다.


## 4.3 누수는 정규화에서만 나지 않습니다

In [5]:
def top_k_features(x, y, k):
    """타깃과 상관이 큰 특징 k개를 고른다. 흔한 전처리입니다."""
    c = np.array([np.corrcoef(x[:, j], y)[0, 1] for j in range(x.shape[1])])
    return np.argsort(-np.nan_to_num(np.abs(c)))[:k]


N, F, K = 300, 400, 10
bad, ok = [], []
seeds = range(3 if dlbook.smoke.is_smoke() else 8)

for sd in seeds:
    rng = np.random.default_rng(sd)
    X = rng.normal(size=(N, F)).astype("float32")
    Y = rng.integers(0, 2, N).astype("int64")     # ★ 정답이 동전 던지기입니다

    sel = top_k_features(X, Y, K)                 # 틀림 — 전체를 보고 고른다
    bad.append(train(data.split(X[:, sel], Y, seed=sd)))

    s = data.split(X, Y, seed=sd)                 # 맞음 — 나누고 학습만 보고
    j = top_k_features(s.x_train, s.y_train, K)
    ok.append(train(data.Split(s.x_train[:, j], s.y_train,
                               s.x_val[:, j], s.y_val,
                               s.x_test[:, j], s.y_test)))
    print(f"  시드 {sd}   누수 {bad[-1]:.3f}   제대로 {ok[-1]:.3f}", flush=True)

print()
print(f"  {'누수':<8}{np.mean(bad):.3f}")
print(f"  {'제대로':<8}{np.mean(ok):.3f}")
print(f"  {'진실':<8}0.500   ← 정답이 무작위입니다. 맞힐 방법이 없습니다.")
dlbook.record("ch04_leak_select_bad", float(np.mean(bad)))
dlbook.record("ch04_leak_select_ok", float(np.mean(ok)))

print()
print(f"★ **신호가 하나도 없는 데이터에서 {np.mean(bad):.3f} 가 나왔습니다.**")
print(f"  {F}개 잡음 중 우연히 정답과 비슷해 보이는 {K}개를 골랐는데,")
print("  **고를 때 시험 데이터를 봤기 때문에** 그 우연이 시험에서도 통합니다.")
print(f"★ 제대로 하면 {np.mean(ok):.3f} — 있는 그대로 0.5입니다. 이게 정직한 숫자입니다.")

  시드 0   누수 0.567   제대로 0.517


  시드 1   누수 0.617   제대로 0.533


  시드 2   누수 0.617   제대로 0.500


  시드 3   누수 0.650   제대로 0.450


  시드 4   누수 0.550   제대로 0.450


  시드 5   누수 0.683   제대로 0.633


  시드 6   누수 0.517   제대로 0.533


  시드 7   누수 0.550   제대로 0.433



  누수      0.594
  제대로     0.506
  진실      0.500   ← 정답이 무작위입니다. 맞힐 방법이 없습니다.
ch04_leak_select_bad = 0.5938
ch04_leak_select_ok = 0.5062

★ **신호가 하나도 없는 데이터에서 0.594 가 나왔습니다.**
  400개 잡음 중 우연히 정답과 비슷해 보이는 10개를 골랐는데,
  **고를 때 시험 데이터를 봤기 때문에** 그 우연이 시험에서도 통합니다.
★ 제대로 하면 0.506 — 있는 그대로 0.5입니다. 이게 정직한 숫자입니다.


## 정리

| 정규화 누수 | 시험 정확도 |
|---|:--:|
| 틀린 순서 — 전체로 정규화하고 나눈다 | 0.950 |
| 맞는 순서 — 나누고 학습 통계로만 | 0.950 |
| **차이** | **0.000** |

| 특징 선택 누수 | 시험 정확도 |
|---|:--:|
| 틀린 순서 — 전체를 보고 고른다 | **0.594** |
| 맞는 순서 — 나누고 학습만 보고 고른다 | 0.506 |
| **진실** | **0.500** |

- **정규화 누수는 거의 아무 일도 일으키지 않습니다.** 평균과 표준편차는
  표본 몇백 개면 안정적이라, 전체로 재나 학습 데이터로만 재나 같습니다.
  데이터를 100개로 줄여도 차이가 0.000입니다.
- **그런데 특징 선택 누수는 재앙입니다.** 정답이 동전 던지기인 데이터에서
  0.594가 나옵니다. 논문 한 편이 여기서 만들어집니다.
- **둘의 차이는 「무엇을 골랐느냐」입니다.** 정규화는 값을 옮길 뿐이지만,
  선택은 **시험 데이터를 보고 결정을 내립니다.** 결정이 시험 데이터를
  보고 내려지는 순간 그 시험은 시험이 아닙니다.
- 그래서 §4.3의 규칙은 이렇게 읽는 것이 맞습니다 —
  **"정규화를 나중에 하라"가 아니라 "학습 데이터를 보고 정한 것은 전부
  그대로 시험에 적용하라"**. 결측치 대치의 중앙값, 범주형 인코딩의 범주
  목록, 이상치 절단의 경계, 그리고 **특징 선택**.

### 연습

1. 특징 개수 `F` 를 400에서 40으로 줄이면 누수의 크기는 어떻게 됩니까.
   왜 그렇습니까.
2. 표본 `N` 을 300에서 3,000으로 늘리면 어떻게 됩니까.
3. 7장을 미리 보십시오. **모델을 여러 번 골라 시험 성능이 가장 좋은 것을
   고르는 것**도 같은 종류의 누수입니다. 어디가 같습니까.
4. 정규화 누수가 **실제로 위험해지는 경우**를 하나 만들어 보십시오.
   (힌트: 시간이 흐르는 데이터. 미래가 과거로 새어 듭니다)